## Fine-tuning Stable Diffusion XL with DreamBooth and LoRA 🧨

In this notebook, we fine-tune [Stable Diffusion XL (SDXL)](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion/stable_diffusion_xl) with [DreamBooth](https://huggingface.co/docs/diffusers/main/en/training/dreambooth) and [LoRA](https://huggingface.co/docs/diffusers/main/en/training/lora).

SDXL consists of a much larger U-Net and two text encoders that make the cross-attention context quite larger than the previous variants.

So, to fine-tune such a large model, we will make use of several tricks such as gradient checkpointing, mixed-precision, and 8-bit Adam.

## Setup 🪓

In [ ]:
# Check if GPUs are available
!nvidia-smi

In [ ]:
# Install dependencies
!pip install bitsandbytes transformers accelerate peft -q

Install `diffusers` from its github repository.

In [ ]:
!pip install git+https://github.com/huggingface/diffusers.git -q

Download the `diffusers` SDXL DreamBooth-LoRA training script.

In [ ]:
!wget https://raw.githubusercontent.com/huggingface/diffusers/refs/heads/main/examples/dreambooth/train_dreambooth_lora_sdxl.py

In [ ]:
# Define training configuration for logging to W & B
config = {
    "checkpointing_steps":         100,
    "dataset":                     "ajesteves/caoserraestrela",
    "gradient_accumulation_steps": 4,
    "gradient_checkpointing":      True,
    "instance_prompt":             "a photo of CSESTRELA dog",
    "learning_rate":               0.0001,
    "lr_scheduler":                "constant",
    "lr_warmup_steps":             0,
    "max_train_steps":             1200,
    "mixed_precision":             "fp16",
    "pretrained_model":            "stabilityai/stable-diffusion-xl-base-1.0",
    "pretrained_vae_model":        "madebyollin/sdxl-vae-fp16-fix",
    "resolution":                  1024,
    "snr_gamma":                   5,
    "train_batch_size":            2,
    "use_8bit_adam":               True
}

Connect to Weights and Biases

In [ ]:
import wandb
from   kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
key          = user_secrets.get_secret("WANDB")

!wandb login $key

wandb.init(
   project = "SDXL-dreambooth-lora",
   entity  = "ajesteves",
   config  = config,
#   resume = True,
#   id     = "?????" # W & B ID to resume from a previous run
)

## Dataset 🐶

**Get the training data!**

For this example, we will download images from an Hugging Face dataset.

If the images are saved locally, and/or we want to add BLIP generated captions, pick option 1 or 2 below.



**Option 1:** upload example images from our local files

In [ ]:
'''
import os
from google.colab import files

# pick a name for the image folder
local_dir = "./dog/" #@param
os.makedirs(local_dir)
os.chdir(local_dir)

# choose and upload local images into the newly created directory
uploaded_images = files.upload()
os.chdir("/content") # back to parent directory
'''

**Option 2:** download example images from the HF hub

In [ ]:
from huggingface_hub import snapshot_download

local_dir = "./csestrela/"
snapshot_download(
    "ajesteves/caoserraestrela",
    local_dir       = local_dir, repo_type="dataset",
    ignore_patterns = ".gitattributes",
)

Preview the images.

In [ ]:
from PIL import Image

def image_grid(imgs, rows, cols, resize=256):

    if resize is not None:
        imgs = [img.resize((resize, resize)) for img in imgs]
    w, h = imgs[0].size
    grid = Image.new("RGB", size=(cols * w, rows * h))
    grid_w, grid_h = grid.size

    for i, img in enumerate(imgs):
        grid.paste(img, box=(i % cols * w, i // cols * h))
    return grid

In [ ]:
import glob

# change path to display images from our local folder
img_paths = "./csestrela/*.jpg"
imgs      = [Image.open(path) for path in glob.glob(img_paths)]
num_imgs_to_preview = 5

image_grid(imgs[:num_imgs_to_preview], 1, num_imgs_to_preview)

### Generate custom captions with BLIP

Load BLIP to auto caption the images.

In [ ]:
import requests
import torch
from   transformers import AutoProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the BLIP processor and the BLIP captioning model
blip_processor = AutoProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model     = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch.float16
).to(device)

# A captioning utility function
def caption_images(input_image):
    inputs       = blip_processor(images=input_image, return_tensors="pt").to(device, torch.float16)
    pixel_values = inputs.pixel_values

    generated_ids     = blip_model.generate(pixel_values=pixel_values, max_length=50)
    generated_caption = blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return generated_caption

In [ ]:
import glob
from   PIL import Image

# Create a list of (path, PIL.Image) pairs
local_dir      = "./csestrela/"
imgs_and_paths = [(path,Image.open(path)) for path in glob.glob(f"{local_dir}*.jpg")]

Now, let us add our concept token identifier, such as "TOK", to each caption using a caption prefix.
One can change the prefix according to the concept we are training on!
- for this example we can use "a photo of TOK," other options include:
    - For styles - "In the style of TOK"
    - For faces  - "photo of a TOK person"
- You can add additional identifiers to the prefix that can help steer the model in the right direction.
-- e.g. for this example, instead of "a photo of TOK" we can use "a photo of TOK dog" or "a photo of TOK corgi dog".

In our case, TOK will be CSESTRELA.

In [ ]:
import json

caption_prefix = "a photo of CSESTRELA dog, " #@param

with open(f'{local_dir}metadata.jsonl', 'w') as outfile:
  for img in imgs_and_paths:
      caption = caption_prefix + caption_images(img[1]).split("\n")[0]
      entry   = {"file_name":img[0].split("/")[-1], "prompt": caption}
      json.dump(entry, outfile)
      outfile.write('\n')

Free some memory.

In [ ]:
import gc

# delete the BLIP pipelines and free up some memory
del blip_processor, blip_model
gc.collect()
torch.cuda.empty_cache()

## Prepare for training 💻

Initialize `accelerate`.

In [ ]:
import locale

locale.getpreferredencoding = lambda: "UTF-8"

!accelerate config default

### Log into our Hugging Face account
Pass [our **write** access token](https://huggingface.co/settings/tokens) so that we can push the trained checkpoints to the Hugging Face Hub:

In [ ]:
from   kaggle_secrets  import UserSecretsClient
from   huggingface_hub import notebook_login
import os

user_secrets = UserSecretsClient()

# Set your HF token and username as environment variables
os.environ["HF_TOKEN"] = user_secrets.get_secret("HUGGINGFACE")

# Replace with your username)
os.environ["HF_USERNAME"] = "ajesteves"

notebook_login()

## Train SDXL 🔬

#### Set the hyperparameters ⚡

To ensure we can combine DreamBooth with LoRA on a heavy pipeline like Stable Diffusion XL, we are using:

* Gradient checkpointing: `--gradient_accumulation_steps`
* An 8-bit version of the Adam optimizer: `--use_8bit_adam`
* Training with mixed-precision: `--mixed-precision="fp16"`

### Launch training 🚀🚀🚀

To allow for custom captions, we need to install the `datasets` library, but we can skip that if we want to train solely  with `--instance_prompt`.
In that case, specify `--instance_data_dir` instead of `--dataset_name`

In [ ]:
!pip install datasets -q

 - Use `--output_dir` to specify our LoRA model repository name.
 - Use `--caption_column` to specify the name of the caption column in our dataset. In this example, we used "prompt" to save our captions in the  metadata file, but one can change this according to our needs.

In [ ]:
## !/usr/bin/env bash
!accelerate launch train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path      "stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path  "madebyollin/sdxl-vae-fp16-fix" \
  --dataset_name        "ajesteves/caoserraestrela" \
  --output_dir          "csestrela_lora" \
  --mixed_precision     "fp16" \
  --instance_prompt     "a photo of CSESTRELA dog" \
  --resolution          1024 \
  --train_batch_size    2 \
  --gradient_accumulation_steps  4 \
  --gradient_checkpointing \
  --learning_rate       1e-4 \
  --snr_gamma           5.0 \
  --lr_scheduler        "constant" \
  --lr_warmup_steps     0 \
  --use_8bit_adam \
  --max_train_steps     1200 \
  --checkpointing_steps 100 \
  --seed                "123" \
  --report_to           "wandb"

### Save the model to the HF hub and check it out 🔥

In [ ]:
from huggingface_hub import whoami
from pathlib         import Path

#@markdown make sure the `output_dir` specified here is the same as the one used for training
output_dir = "csestrela_lora" #@param
username   = whoami(token=Path("/root/.cache/huggingface/"))["name"]
repo_id    = f"{username}/{output_dir}"

In [ ]:
# @markdown Sometimes training finishes succesfuly, i.e., a **.safetensores** file with the LoRA weights saved properly to our local `output_dir`) but there is not enough RAM in the free tier to push the model to the HF hub 🙁
# @markdown
# @markdown To mitigate this, run this cell with our training arguments to make sure our model is uploaded! 🤗

# push to the HF hub🔥
from train_dreambooth_lora_sdxl import save_model_card
from huggingface_hub            import upload_folder, create_repo

repo_id = create_repo(repo_id, exist_ok=True).repo_id

# change the params below according to the training arguments
save_model_card(
    repo_id            = repo_id,
    images             = [],
    base_model         = "stabilityai/stable-diffusion-xl-base-1.0",
    train_text_encoder = False,
    instance_prompt    = "a photo of CSESTRELA dog",
    validation_prompt  = None,
    repo_folder        = output_dir,
    vae_path           = "madebyollin/sdxl-vae-fp16-fix",
    use_dora           = False,
)

upload_folder(
    repo_id=repo_id,
    folder_path     = output_dir,
    commit_message  = "End of training",
    ignore_patterns = ["step_*", "epoch_*"],
)

In [ ]:
from IPython.display import display, Markdown

link_to_model = f"https://huggingface.co/{repo_id}"

display(Markdown("### The model has finished training.\nAccess it here: {}".format(link_to_model)))

Let us generate some images with the fine-tuned model.

## Inference 🐕

In [ ]:
import torch
from diffusers import DiffusionPipeline, AutoencoderKL

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae             = vae,
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True,
)
pipe.load_lora_weights(repo_id)
_ = pipe.to("cuda")

In [ ]:
prompt = "a photo of CSESTRELA dog in a bucket at the beach" # @param

image  = pipe(prompt=prompt, num_inference_steps=25).images[0]

display(image)